In [1]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [2]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
v2 = model.encode(q2)

In [9]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [10]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
v1.dot(dv) # Compare the similarity between the first question and a possible answer

np.float32(0.32332397)

In [12]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [ ]:
v2.dot(dv) # Compare the similarity between unrealated question and an answer - almost orthogonal vectors (not simillar)

np.float32(0.019730574)

First lets get the data before embedding it

In [3]:
from ingest import load_faq_data
documents = load_faq_data()
documents[10]

{'id': '2b5ff70c77',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to enroll in the course before submitting homework?',
 'answer': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'}

In [5]:
texts=[]
for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

len(texts)

1406

Embedding the data to vectors

In [6]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/29 [00:00<?, ?it/s]

1406

In [ ]:
import numpy as np

# Turning documents vectors to matrix. Each row is a document
X = np.array(vectors)

In [ ]:
""" scores = []
for i in range(len(vectors)):
    score = v1.dot(vectors[i]) # Checking simillarity score with v1 query
    scores.append(score) """

scores = X.dot(v1) # Extremely increasing optimization of the above for loop

In [ ]:
idx = np.argmax(scores) # Getting most similar ones with highest results
idx, scores[idx] # Document index and its score

(np.int64(1004), np.float32(0.762941))

In [17]:
documents[1004]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

We will want to get the top 5 documents most similar to our query

In [ ]:
""" top5 = np.argsort(scores)[-5:] # Getting indexes of top 5 most similar documents
top5 = top5[::-1] # Changing order so the first element is the most similar one with the highest """

top5 = np.argsort(-scores)[:5] # Equivalent to the above lines
top5

array([1004,  751,   29,  605, 1009])

In [36]:
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [37]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [ ]:
from minsearch import VectorSearch

# Using minsearch to do Vector Search
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [42]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'} )

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project s